# Phase 2 — Nanda baseline (5 seeds) + identification des fréquences caractéristiques

Objectif : run la config canonique de Nanda 2023 (`AdamW lr=1e-3, wd=1.0`) sur 5 seeds, et identifier les fréquences caractéristiques du circuit Fourier final.

Toutes les fonctions de visualisation sont dans `viz_analysis.py`. Pour les prochaines étapes (Fast AdamW, EGD, Muon...), on réutilisera les mêmes fonctions.

## Setup

In [ ]:
#!pip install -q torch plotly einops

In [ ]:
from pathlib import Path
import numpy as np
import torch as t
import matplotlib.pyplot as plt
import importlib, pipeline, model, viz_analysis
importlib.reload(pipeline); importlib.reload(model); importlib.reload(viz_analysis)

from model        import Config
from pipeline     import OptimizerSpec, run_multi_seed
from viz_analysis import (
    load_seeds, compute_grokking_markers, per_seed_stats,
    plot_curves_band, plot_fourier_losses_band,
    plot_freq_mass_heatmap, plot_sparsity,
    plot_fourier_components, plot_fourier_components_per_seed, identify_key_freqs,
)

%matplotlib inline
plt.rcParams.update({'figure.facecolor':'white','figure.dpi':100,'savefig.dpi':200,'font.size':11})
t.manual_seed(0); np.random.seed(0)

ROOT      = Path.cwd()
SAVE_ROOT = ROOT / 'runs' / 'phase2' / 'adamw_nanda'
FIG_DIR   = ROOT / 'figs_phase2'
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)
print(f'Save root : {SAVE_ROOT}')
print(f'Fig dir   : {FIG_DIR}')

## Config + spec Nanda canonique

In [ ]:
BASE_CONFIG = Config(
    p=113, d_model=128, d_mlp=512, num_heads=4, n_ctx=3,
    act_type='ReLU', frac_train=0.3,
    num_epochs=25_000,
    seed=0,
)

NANDA_SPECS = [
    OptimizerSpec(
        name='adamw',
        lr=1e-3, weight_decay=1.0,
        extra={'betas': (0.9, 0.98)},
    ),
]

SEEDS         = [0, 1, 2, 3, 4]
EVAL_EVERY    = 50
FOURIER_EVERY = 100
WARMUP_STEPS  = 10

print(f'  Specs   : {[s.describe() for s in NANDA_SPECS]}')
print(f'  Seeds   : {SEEDS}')
print(f'  Epochs  : {BASE_CONFIG.num_epochs}')
print(f'  Eval    : every {EVAL_EVERY}')
print(f'  Fourier : every {FOURIER_EVERY}')

## Run 5 seeds — resume-aware

In [ ]:
missing = [s for s in SEEDS if not (SAVE_ROOT / f'seed{s}' / 'history.json').exists()]
done    = [s for s in SEEDS if s not in missing]
if done:    print(f'  already done : {done}')
if missing:
    print(f'  will run     : {missing}')
    _ = run_multi_seed(
        BASE_CONFIG, NANDA_SPECS, seeds=missing,
        label_prefix='adamw_nanda',
        save_root=str(SAVE_ROOT),
        eval_every=EVAL_EVERY,
        fourier_every=FOURIER_EVERY,
        warmup_steps=WARMUP_STEPS,
        verbose_every=2_500,
        verbose_build=False,
    )
else:
    print('  All 5 seeds done — skipping training.')

## Reload + stats

In [ ]:
runs, runs_dict = load_seeds(SAVE_ROOT, SEEDS)
markers = compute_grokking_markers(runs)
stats   = per_seed_stats(runs)

print(f'═══ AdamW Nanda baseline — n={len(runs)} seeds ═══')
print(f'  mem     = {markers["mem"]}    (first epoch where MEAN train_acc >= 0.99)')
print(f'  circuit = {markers["circuit"]}    (first epoch where MEAN wl_top5 >= 0.5)')
print(f'  grok    = {markers["grok"]}    (first epoch where MEAN test_acc >= 0.99)')
if markers['grok'] and markers['mem']:
    print(f'  gap         = {markers["grok"] - markers["mem"]}    (grokking delay)')
print(f'')
print(f'  --- per-seed grok times (reference) ---')
print(f'  list   : {stats["list"]}')
print(f'  median : {stats["median"]}')
print(f'  IQR    : {stats["iqr"]}')
print(f'  range  : {stats["range"]}')

## 1 — Train/test/L2 curves

In [ ]:
plot_curves_band(
    runs,
    title='Nanda baseline — train/test loss + accuracy + L2',
    markers=markers,
    save_path=FIG_DIR / 'nanda_curves.png',
)

### Sanity check — la bold curve est bien la MEAN, pas la median

À epochs choisis, on imprime : valeurs par seed, mean, median. Compare avec ce que tu vois sur le plot.

In [ ]:
import numpy as np
from viz_analysis import stack

# Stack test_loss across seeds (n_seeds, T)
ep, stk = stack(runs, 'test_loss')

print(f'═══ test_loss : MEAN vs MEDIAN at chosen epochs ═══')
print(f'{"epoch":>7} | {"per-seed values":<50} | {"MEAN":>12} | {"MEDIAN":>12}')
print('-' * 100)
for target_ep in [150, 1000, 3000, 5000, 7000, 9000, 11000, 15000, 25000]:
    idx = int(np.argmin(np.abs(ep - target_ep)))
    vals = stk[:, idx]
    mean_v   = float(vals.mean())
    median_v = float(np.median(vals))
    vals_str = '[' + ', '.join(f'{v:.2e}' for v in vals) + ']'
    print(f'{int(ep[idx]):>7} | {vals_str:<50} | {mean_v:>12.4e} | {median_v:>12.4e}')

print(f'\n→ la bold red curve sur le plot affiche la colonne MEAN à chaque epoch.')
print(f'  Si la curve passe par MEDIAN au lieu de MEAN, signaler un bug.\n')

# Even more direct : print test_acc, which is bounded [0,1] and easier to inspect
print(f'═══ test_acc : MEAN vs MEDIAN ═══')
print(f'{"epoch":>7} | {"per-seed values":<50} | {"MEAN":>8} | {"MEDIAN":>8}')
print('-' * 90)
_, stk_acc = stack(runs, 'test_acc')
for target_ep in [150, 1000, 3000, 5000, 7000, 9000, 11000, 15000, 25000]:
    idx = int(np.argmin(np.abs(ep - target_ep)))
    vals = stk_acc[:, idx]
    vals_str = '[' + ', '.join(f'{v:.3f}' for v in vals) + ']'
    print(f'{int(ep[idx]):>7} | {vals_str:<50} | {vals.mean():>8.4f} | {np.median(vals):>8.4f}')

## 2 — Fourier progress losses

In [ ]:
plot_fourier_losses_band(
    runs,
    title='Nanda baseline — Fourier progress losses',
    markers=markers,
    save_path=FIG_DIR / 'nanda_fourier_losses.png',
)

## 3 — Frequency mass heatmap (timeline d'émergence)

In [ ]:
plot_freq_mass_heatmap(
    runs,
    title='Nanda baseline — frequency mass evolution',
    markers=markers,
    save_path=FIG_DIR / 'nanda_freq_masses.png',
)

## 4 — Sparsité spectrale (gini, entropy, top-5)

In [ ]:
plot_sparsity(
    runs,
    title='Nanda baseline — spectral sparsity',
    markers=markers,
    save_path=FIG_DIR / 'nanda_sparsity.png',
)

## 5 — Fourier components bar chart (Nanda-style) + identification key freqs

Charge `model.pt` de chaque seed, projette W_E et W_U sur la base Fourier réelle de Z_p. Le plot est identique au plot original de Nanda 2023.

In [ ]:
per_seed_components = plot_fourier_components(
    save_root=SAVE_ROOT,
    seeds=SEEDS,
    config=BASE_CONFIG,
    title='Phase 2 — Nanda baseline (AdamW lr=1e-3, wd=1.0)',
    save_path=FIG_DIR / 'nanda_fourier_components.png',
)

## Explication de l'identification des key freqs

Pour chaque seed, à partir du `model.pt` final :

**Étape 1 — Projection sur la base Fourier réelle de Z_p**

W_E (embedding) et W_U (unembed) ont shape `(d_model, d_vocab=114)`. On garde les `p=113` premières colonnes.

Pour chaque fréquence k = 1, ..., p//2 = 56 :
- `we_cos[k] = ||W_E[:, :p] @ cos_basis[k]||₂`   (norme L2 sur d_model)
- `we_sin[k] = ||W_E[:, :p] @ sin_basis[k]||₂`
- `wu_cos[k] = ||W_U[:, :p] @ cos_basis[k]||₂`
- `wu_sin[k] = ||W_U[:, :p] @ sin_basis[k]||₂`

**Étape 2 — Score combiné par freq**
```
score[k] = we_cos[k] + we_sin[k] + wu_cos[k] + wu_sin[k]
```
Mesure : à quel point la fréquence k est utilisée par W_E ET W_U, sin ET cos.

**Étape 3 — Top-k par seed** : les `k` freqs avec les plus hauts scores.

**Étape 4 — Agrégation cross-seeds**
- `consensus` : freqs dans le top-k de TOUS les seeds (signature universelle)
- `majority` : freqs dans le top-k de la majorité des seeds (≥ n/2 + 1)

**Pré-filtrage** : on n'inclut que les seeds qui ont bien grokké (`final_test_acc >= 0.99`) pour éviter que des circuits incomplets pollueront l'identification.

In [ ]:
# ──── 1) Filter seeds qui ont bien grokké (final_test_acc >= 0.99) ────────
CONVERGED_THRESH = 0.99
converged_seeds = []
print('═══ Convergence check ═══')
for r in runs:
    h = r['history']
    final_acc = h['test_acc'][-1]
    final_loss = h['test_loss'][-1] if h['test_loss'][-1] == h['test_loss'][-1] else float('nan')
    status = '✓' if final_acc >= CONVERGED_THRESH else '✗ NON CONVERGÉ'
    print(f'  seed {r["seed"]} : final_test_acc = {final_acc:.4f}, final_test_loss = {final_loss:.2e}   {status}')
    if final_acc >= CONVERGED_THRESH:
        converged_seeds.append(r['seed'])

print(f'\n→ {len(converged_seeds)}/{len(runs)} seeds ont convergé : {converged_seeds}')

if len(converged_seeds) < 3:
    print('⚠ moins de 3 seeds convergés — augmenter num_epochs ou vérifier la config')

# ──── 2) Re-charger per_seed_components SEULEMENT pour les seeds convergés ────
per_seed_converged = {s: per_seed_components[s] for s in converged_seeds if s in per_seed_components}

# ──── 3) Identification des key freqs sur seeds convergés uniquement ─────
key_freqs = identify_key_freqs(per_seed_converged, k=5)

print(f'\n═══ Identification des fréquences caractéristiques (n={len(converged_seeds)} seeds convergés) ═══')
print(f'\n  Top-5 freqs par seed :')
for s, freqs in key_freqs['per_seed'].items():
    print(f'    seed {s} : {freqs}')
print(f'\n  Consensus (top-5 dans TOUS les seeds convergés) : {key_freqs["consensus"]}')
print(f'  Majority  (top-5 dans >=3 seeds convergés)      : {key_freqs["majority"]}')

# ──── 4) Si consensus est vide, essayer top-10 (plus permissif) ──────────
if not key_freqs['consensus']:
    key_freqs_10 = identify_key_freqs(per_seed_converged, k=10)
    print(f'\n  → consensus vide en top-5. Avec top-10 :')
    print(f'    Consensus (top-10) : {key_freqs_10["consensus"]}')
    print(f'    Majority  (top-10) : {key_freqs_10["majority"]}')

## 6 — Bar chart par seed (visualisation de la variabilité inter-seed)

Un panel par seed (W_E à gauche, W_U à droite). Permet de voir si les key freqs identifiées sont vraiment partagées, ou si chaque seed apprend un circuit différent.

In [ ]:
_per_seed = plot_fourier_components_per_seed(
    save_root=SAVE_ROOT,
    seeds=converged_seeds,   # uniquement les seeds qui ont grokké
    config=BASE_CONFIG,
    title='Phase 2 — Nanda baseline — per-seed Fourier components',
    save_path=FIG_DIR / 'nanda_fourier_components_per_seed.png',
)